In [2]:
#!/usr/bin/env python
from __future__ import annotations

import os
import json
from datetime import datetime, timezone
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt

import rioxarray
from sqlalchemy import create_engine, text
from shapely.geometry import shape
from shapely.ops import transform as shp_transform
from pyproj import Transformer
from pystac_client import Client
import planetary_computer

# ----------------------------------------------------------------------
# CONFIGURAÇÕES GERAIS
# ----------------------------------------------------------------------
# Planetary Computer STAC
PC_STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

# PostGIS
PG_USER = "postgres"
PG_PASS = "postgres"
PG_HOST = "localhost"
PG_PORT = "5432"
PG_DB   = "geologia"

PG_CONN_STR = (
    f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

# Folha alvo e data alvo aerogeofísica (para ranking de proximidade temporal)
FOLHA_CODIGO = "SB21_ZA_II2_NW"
ASTER_TARGET_DATE_STR = "2008-07-15"  # data alvo (ex. aquisição aerogeofísica)


# ----------------------------------------------------------------------
# UTILITÁRIOS GERAIS
# ----------------------------------------------------------------------
def parse_target_date(date_str: str) -> datetime:
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    return dt.replace(tzinfo=timezone.utc)


def item_datetime(item) -> datetime:
    """
    Extrai datetime (timezone-aware) de um pystac.Item.
    """
    if getattr(item, "datetime", None) is not None:
        dt = item.datetime
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt

    props = getattr(item, "properties", {}) or {}
    dt_str = props.get("datetime") or props.get("start_datetime")
    if dt_str is None:
        raise RuntimeError(f"Item {getattr(item, 'id', '?')} sem campo datetime.")

    if dt_str.endswith("Z"):
        dt_str = dt_str.replace("Z", "+00:00")
    dt = datetime.fromisoformat(dt_str)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt


def get_cloud_cover(item) -> Optional[float]:
    """
    Tenta ler 'eo:cloud_cover' ou 's2:cloud_cover' das propriedades.
    """
    props = getattr(item, "properties", {}) or {}
    for key in ("eo:cloud_cover", "s2:cloud_cover"):
        if key in props and props[key] is not None:
            try:
                return float(props[key])
            except Exception:
                continue
    return None


def get_pc_client() -> Client:
    """
    Client do Planetary Computer que já assina os assets (SAS) in-place.
    """
    return Client.open(
        PC_STAC_URL,
        modifier=planetary_computer.sign_inplace,
    )


# ----------------------------------------------------------------------
# ACESSO AO POSTGIS E GEOMETRIA DA FOLHA
# ----------------------------------------------------------------------
def get_folha_geom_geojson(
    codigo_folha: str,
    conn_str: str = PG_CONN_STR,
    schema: str = "carto",
    table: str = "folhas_cartograficas",
) -> Dict[str, Any]:
    """
    Retorna a geometria da folha em WGS84 (EPSG:4326) como dict GeoJSON.
    """
    engine = create_engine(conn_str)
    sql = text(
        f"""
        SELECT ST_AsGeoJSON(ST_Transform(geom, 4326)) AS geom_json
        FROM {schema}.{table}
        WHERE codigo = :codigo
        LIMIT 1;
        """
    )
    with engine.begin() as conn:
        row = conn.execute(sql, {"codigo": codigo_folha}).fetchone()

    if row is None or row.geom_json is None:
        raise RuntimeError(f"Não foi encontrada geometria para a folha '{codigo_folha}'")

    return json.loads(row.geom_json)


# ----------------------------------------------------------------------
# FUNÇÕES GEOMÉTRICAS PARA COBERTURA (ASTER / S2)
# ----------------------------------------------------------------------
def build_local_equal_area_transformer(geom_geojson: Dict[str, Any]) -> Transformer:
    """
    Projeção local equivalente (LAEA) centrada na folha para cálculo de área/cobertura.
    """
    geom = shape(geom_geojson)
    lon0, lat0 = geom.centroid.x, geom.centroid.y
    laea_proj = (
        f"+proj=laea +lat_0={lat0} +lon_0={lon0} "
        "+datum=WGS84 +units=m +no_defs"
    )
    return Transformer.from_crs("EPSG:4326", laea_proj, always_xy=True)


def coverage_fraction(
    folha_geom_geojson: Dict[str, Any],
    item_geom_geojson: Dict[str, Any],
    transformer: Transformer,
) -> float:
    """
    Fração de cobertura (área de interseção / área da folha) em projeção equivalente.
    """
    folha_geom = shape(folha_geom_geojson)
    item_geom = shape(item_geom_geojson)

    project = transformer.transform
    folha_proj = shp_transform(project, folha_geom)
    item_proj = shp_transform(project, item_geom)

    inter = folha_proj.intersection(item_proj)
    if folha_proj.is_empty or inter.is_empty:
        return 0.0
    return inter.area / folha_proj.area


# ----------------------------------------------------------------------
# CLIP RASTER E MÉTRICAS (NODATA / NUVEM)
# ----------------------------------------------------------------------
def clip_raster_to_folha(href: str, folha_geom_geojson: dict):
    """
    Abre o raster indicado por href e recorta para a geometria da folha (EPSG:4326).
    Retorna um DataArray rioxarray recortado.
    """
    da = rioxarray.open_rasterio(href, masked=True)
    da_clip = da.rio.clip([folha_geom_geojson], crs="EPSG:4326", drop=True)
    return da_clip


def compute_nodata_fraction(da_clip):
    """
    Fração de pixels nodata no recorte.
    Se houver várias bandas, nodata = todas as bandas NaN.
    """
    arr = da_clip.values
    if arr.ndim == 3:
        nodata_mask = np.all(np.isnan(arr), axis=0)
    else:
        nodata_mask = np.isnan(arr)
    frac = nodata_mask.sum() / nodata_mask.size
    return frac, nodata_mask


def compute_scl_cloud_fraction(da_scl_clip):
    """
    A partir de um SCL recortado (S2 L2A), calcula:
      - fração de nodata (código 0)
      - fração de nuvem/sombra/neve (códigos {3,8,9,10,11})
    """
    scl = da_scl_clip.values
    if scl.ndim == 3:
        scl = scl[0, :, :]
    scl = scl.astype("int16")

    total = scl.size
    nodata_frac = np.sum(scl == 0) / total
    cloud_mask = np.isin(scl, [3, 8, 9, 10, 11])
    cloud_frac = np.sum(cloud_mask) / total

    return nodata_frac, cloud_frac, scl, cloud_mask


# ----------------------------------------------------------------------
# BUSCA ASTER "SEM NUVEM" (SCENE-LEVEL) PARA UMA FOLHA ESPECÍFICA
# ----------------------------------------------------------------------
def search_aster_cloudfree_for_folha(
    codigo_folha: str,
    aster_target_date_str: str = ASTER_TARGET_DATE_STR,
    collection_id: str = "aster-l1t",
    search_datetime: str = "2000-01-01/2015-12-31",
    min_coverage: float = 0.99,
    max_cloud: float = 5.0,
    max_items: int = 2000,
) -> Tuple[Optional[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Busca cenas ASTER L1T que intersectam a folha, com boa cobertura e baixa nuvem
    (via metadado 'eo:cloud_cover'), e retorna a melhor (e lista de candidatos).

    Critério de ordenação:
      1) cloud_cover crescente
      2) |data - ASTER_TARGET_DATE| crescente
      3) cobertura decrescente
    """
    folha_geom = get_folha_geom_geojson(codigo_folha)
    folha_shape = shape(folha_geom)
    print(
        f"[ASTER] Folha '{codigo_folha}' carregada. "
        f"Área WGS84 (aprox): {folha_shape.area:.6f} (graus²)"
    )

    transformer = build_local_equal_area_transformer(folha_geom)
    client = get_pc_client()

    print(f"[ASTER] Buscando itens em '{collection_id}'...")
    search = client.search(
        collections=[collection_id],
        intersects=folha_geom,
        datetime=search_datetime,
        max_items=max_items,
    )
    items = list(search.items())
    print(f"[ASTER] Total de itens retornados (antes de filtros locais): {len(items)}")

    if not items:
        print("[ASTER] Nenhum item retornado pelo STAC.")
        return None, []

    target_date = parse_target_date(aster_target_date_str).date()
    candidates: List[Dict[str, Any]] = []

    for item in items:
        dt = item_datetime(item)
        dt_date = dt.date()
        delta_days = abs((dt_date - target_date).days)

        cloud = get_cloud_cover(item)
        if cloud is None:
            continue
        if cloud > max_cloud:
            continue

        frac = coverage_fraction(
            folha_geom_geojson=folha_geom,
            item_geom_geojson=item.geometry,
            transformer=transformer,
        )
        if frac < min_coverage:
            continue

        candidates.append(
            {
                "id": item.id,
                "datetime": dt.isoformat(),
                "delta_days": delta_days,
                "coverage_fraction": frac,
                "coverage_percent": frac * 100.0,
                "cloud_cover": cloud,
                "item": item,
            }
        )

    if not candidates:
        print(
            "[ASTER] Nenhum item atendeu a:\n"
            f"  coverage >= {min_coverage*100:.1f}% e cloud <= {max_cloud:.2f}%."
        )
        return None, []

    # Ordenação pelo critério definido
    candidates_sorted = sorted(
        candidates,
        key=lambda d: (d["cloud_cover"], d["delta_days"], -d["coverage_fraction"]),
    )

    print("\n[ASTER] Itens candidatos (ordenados):")
    for c in candidates_sorted:
        print(
            f"  id={c['id']}, "
            f"datetime={c['datetime']}, "
            f"delta_days={c['delta_days']}, "
            f"coverage={c['coverage_percent']:.2f}%, "
            f"cloud={c['cloud_cover']:.2f}%"
        )

    best = candidates_sorted[0]
    print(
        "\n[ASTER] Melhor item:\n"
        f"  id={best['id']}\n"
        f"  datetime={best['datetime']}\n"
        f"  delta_days={best['delta_days']}\n"
        f"  coverage={best['coverage_percent']:.2f}%\n"
        f"  cloud={best['cloud_cover']:.2f}%"
    )
    return best, candidates_sorted


# ----------------------------------------------------------------------
# BUSCA S2 L2A "SEM NUVEM" VIA SCL PARA A MESMA FOLHA E PRÓXIMO À ASTER
# ----------------------------------------------------------------------
def search_s2_cloudfree_for_folha_given_aster(
    codigo_folha: str,
    aster_datetime: datetime,
    search_datetime: str = "2015-06-23/2100-01-01",
    meta_max_cloud: float = 80.0,  # filtro inicial no metadado
    max_items: int = 200,
) -> Tuple[Optional[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Busca cenas Sentinel-2 L2A que intersectam a folha, recorta o SCL para a folha
    e calcula fração de nuvem/sombra/neve. Escolhe a cena com:

      1) menor scl_cloud_frac (nuvem dentro da folha)
      2) menor |data_S2 - data_ASTER|
      3) menor scl_nodata_frac
      4) menor meta_cloud (eo:cloud_cover)

    Retorna a melhor cena e a lista de candidatos com métricas.
    """
    folha_geom = get_folha_geom_geojson(codigo_folha)
    folha_shape = shape(folha_geom)
    print(
        f"[S2] Folha '{codigo_folha}' carregada. "
        f"Área WGS84 (aprox): {folha_shape.area:.6f} (graus²)"
    )

    client = get_pc_client()
    print(f"[S2] Buscando itens em 'sentinel-2-l2a'...")
    search = client.search(
        collections=["sentinel-2-l2a"],
        intersects=folha_geom,
        datetime=search_datetime,
        max_items=max_items,
    )
    items = list(search.items())
    print(f"[S2] Total de itens retornados (antes de filtros locais): {len(items)}")

    if not items:
        print("[S2] Nenhuma cena S2 retornada pelo STAC.")
        return None, []

    candidates: List[Dict[str, Any]] = []

    for item in items:
        props = item.properties or {}
        meta_cloud = props.get("eo:cloud_cover")
        if meta_cloud is None:
            continue
        try:
            meta_cloud = float(meta_cloud)
        except Exception:
            continue

        if meta_cloud > meta_max_cloud:
            continue

        # Precisa ter SCL para estimar nuvem pixel a pixel
        if "SCL" not in item.assets:
            continue

        scl_href = item.assets["SCL"].href

        try:
            da_scl_clip = clip_raster_to_folha(scl_href, folha_geom)
            scl_nodata_frac, scl_cloud_frac, scl_arr, cloud_mask = compute_scl_cloud_fraction(da_scl_clip)
        except Exception as e:
            print(f"[S2] Erro ao processar SCL do item {item.id}: {e}")
            continue

        dt = item_datetime(item)
        delta_days = abs((dt.date() - aster_datetime.date()).days)

        cand = {
            "id": item.id,
            "datetime": dt.isoformat(),
            "delta_days": delta_days,
            "meta_cloud": meta_cloud,
            "scl_nodata_frac": scl_nodata_frac,
            "scl_cloud_frac": scl_cloud_frac,
            "item": item,
        }
        candidates.append(cand)

        print(
            f"[S2] {item.id} | date={dt.date()} | meta_cloud={meta_cloud:.2f}% | "
            f"scl_cloud_frac={scl_cloud_frac:.4f} | scl_nodata_frac={scl_nodata_frac:.4f} | "
            f"Δt(ASTER)={delta_days} dias"
        )

    if not candidates:
        print("[S2] Nenhuma cena passou pelos filtros (SCL + meta_cloud).")
        return None, []

    # Ordenação pelo critério definido
    candidates_sorted = sorted(
        candidates,
        key=lambda d: (
            d["scl_cloud_frac"],
            d["delta_days"],
            d["scl_nodata_frac"],
            d["meta_cloud"],
        ),
    )

    best = candidates_sorted[0]
    print("\n[S2] Melhor cena segundo SCL dentro da folha e proximidade da ASTER:")
    print(
        f"  id={best['id']}\n"
        f"  datetime={best['datetime']}\n"
        f"  Δt(ASTER)={best['delta_days']} dias\n"
        f"  meta_cloud={best['meta_cloud']:.2f}%\n"
        f"  scl_cloud_frac={best['scl_cloud_frac']:.4f}\n"
        f"  scl_nodata_frac={best['scl_nodata_frac']:.4f}"
    )

    return best, candidates_sorted


# ----------------------------------------------------------------------
# INSPEÇÃO / VISUALIZAÇÃO DO PAR ASTER + S2
# ----------------------------------------------------------------------
def inspect_aster_clip(aster_item, folha_geom_geojson: dict):
    """
    Recorta ASTER (asset VNIR) para a folha, calcula nodata e plota imagem + máscara.
    """
    print("\n[INSPECT ASTER]")

    print(f"  ASTER id: {aster_item.id}")
    print(f"  cloud_cover (metadado): {aster_item.properties.get('eo:cloud_cover')} %")

    assets = aster_item.assets
    print("  Assets ASTER disponíveis:", list(assets.keys()))

    if "VNIR" in assets:
        asset_key = "VNIR"
    else:
        asset_key = list(assets.keys())[0]
        print(f"  Asset 'VNIR' não encontrado, usando '{asset_key}'.")

    href = assets[asset_key].href
    print(f"  Lendo asset '{asset_key}' e recortando para a folha...")

    da_clip = clip_raster_to_folha(href, folha_geom_geojson)
    frac_nodata, nodata_mask = compute_nodata_fraction(da_clip)
    print(f"  Fração de pixels nodata no recorte ASTER: {frac_nodata:.4f}")

    if "band" in da_clip.dims:
        da_plot = da_clip.isel(band=0)
    else:
        da_plot = da_clip

    data = da_plot.values

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    im1 = ax1.imshow(data, origin="upper")
    ax1.set_title("ASTER VNIR recortado - intensidade (banda 1)")
    fig.colorbar(im1, ax=ax1, shrink=0.7)

    im2 = ax2.imshow(nodata_mask, origin="upper")
    ax2.set_title("ASTER VNIR recortado - máscara (True = nodata)")
    fig.colorbar(im2, ax=ax2, shrink=0.7)

    plt.tight_layout()
    plt.show()

    return da_clip, nodata_mask


def inspect_s2_clip(s2_item, folha_geom_geojson: dict):
    """
    Recorta Sentinel-2 L2A (asset 'visual' e SCL) para a folha,
    calcula nodata e nuvem, e plota RGB + máscaras.
    """
    print("\n[INSPECT S2]")

    print(f"  S2 id: {s2_item.id}")
    print(f"  cloud_cover (metadado): {s2_item.properties.get('eo:cloud_cover')} %")
    print("  Assets S2 disponíveis:", list(s2_item.assets.keys()))

    # 1) VISUAL
    if "visual" in s2_item.assets:
        visual_key = "visual"
    else:
        visual_key = list(s2_item.assets.keys())[0]
        print(f"  Asset 'visual' não encontrado, usando '{visual_key}'.")

    href_vis = s2_item.assets[visual_key].href
    print(f"  Lendo asset '{visual_key}' e recortando para a folha...")
    da_vis_clip = clip_raster_to_folha(href_vis, folha_geom_geojson)
    frac_nodata_vis, nodata_mask_vis = compute_nodata_fraction(da_vis_clip)
    print(f"  Fração de nodata no visual recortado: {frac_nodata_vis:.4f}")

    arr = da_vis_clip.values
    if arr.ndim == 3 and da_vis_clip.sizes.get("band", 1) >= 3:
        arr = arr.astype("float32")
        valid = arr[~np.isnan(arr)]
        if valid.size > 0:
            vmin = np.nanpercentile(valid, 2)
            vmax = np.nanpercentile(valid, 98)
            scale = vmax - vmin if vmax > vmin else 1.0
            arr_scaled = (arr - vmin) / scale
            arr_scaled = np.clip(arr_scaled, 0, 1)
        else:
            arr_scaled = np.zeros_like(arr)

        rgb = np.transpose(arr_scaled[:3, :, :], (1, 2, 0))

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        ax1.imshow(rgb)
        ax1.set_title("S2 'visual' recortado (RGB)")

        ax2.imshow(nodata_mask_vis, origin="upper")
        ax2.set_title("S2 'visual' - máscara nodata (True = nodata)")
        plt.tight_layout()
        plt.show()
    else:
        data = da_vis_clip.values
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        im1 = ax1.imshow(data, origin="upper")
        ax1.set_title("S2 recortado - intensidade")
        fig.colorbar(im1, ax=ax1, shrink=0.7)

        im2 = ax2.imshow(nodata_mask_vis, origin="upper")
        ax2.set_title("S2 recortado - máscara nodata")
        fig.colorbar(im2, ax=ax2, shrink=0.7)
        plt.tight_layout()
        plt.show()

    # 2) SCL
    if "SCL" not in s2_item.assets:
        print("[INSPECT S2] Asset 'SCL' não encontrado, não é possível visualizar nuvem.")
        return da_vis_clip, nodata_mask_vis, None, None

    href_scl = s2_item.assets["SCL"].href
    print("  Lendo asset 'SCL' e recortando para a folha...")
    da_scl_clip = clip_raster_to_folha(href_scl, folha_geom_geojson)
    scl_nodata_frac, scl_cloud_frac, scl_arr, cloud_mask = compute_scl_cloud_fraction(da_scl_clip)

    print(f"  Fração de nodata no SCL recortado   : {scl_nodata_frac:.4f}")
    print(f"  Fração de nuvem/sombra/neve (SCL)   : {scl_cloud_frac:.4f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    im1 = ax1.imshow(scl_arr, origin="upper")
    ax1.set_title("S2 SCL recortado (códigos de classificação)")
    fig.colorbar(im1, ax=ax1, shrink=0.7)

    im2 = ax2.imshow(cloud_mask, origin="upper")
    ax2.set_title("S2 SCL - máscara nuvem/sombra/neve")
    fig.colorbar(im2, ax=ax2, shrink=0.7)
    plt.tight_layout()
    plt.show()

    return da_vis_clip, nodata_mask_vis, da_scl_clip, cloud_mask


# ----------------------------------------------------------------------
# MAIN: BUSCA PAR ASTER + S2 E INSPEÇÃO
# ----------------------------------------------------------------------
def main():
    folha = FOLHA_CODIGO
    aster_target = ASTER_TARGET_DATE_STR

    print("=" * 80)
    print(f"[PIPELINE] Folha alvo: {folha}")
    print("=" * 80)

    # 1) ASTER "cloud free" (via metadado, alta cobertura)
    aster_best, aster_candidates = search_aster_cloudfree_for_folha(
        codigo_folha=folha,
        aster_target_date_str=aster_target,
        collection_id="aster-l1t",
        search_datetime="2000-01-01/2015-12-31",
        min_coverage=0.99,
        max_cloud=1.0,          # muito restritivo -> tende a pegar 0% ou ~0%
        max_items=2000,
    )

    if aster_best is None:
        print("[PIPELINE] Nenhuma cena ASTER adequada encontrada. Encerrando.")
        return

    aster_item = aster_best["item"]
    aster_dt = item_datetime(aster_item)

    print(
        f"\n[PIPELINE] Data de referência para S2 = data da ASTER selecionada: "
        f"{aster_dt.isoformat()}"
    )

    # 2) S2 L2A "cloud free" dentro da folha (via SCL) e próximo da data ASTER
    s2_best, s2_candidates = search_s2_cloudfree_for_folha_given_aster(
        codigo_folha=folha,
        aster_datetime=aster_dt,
        search_datetime="2004-01-01/2015-12-31",
        meta_max_cloud=1.0,
        max_items=200,
    )

    if s2_best is None:
        print("[PIPELINE] Nenhuma cena S2 adequada encontrada. Encerrando.")
        return

    s2_item = s2_best["item"]

    # 3) Resumo do par
    print("\n[PAR SELECIONADO ASTER + S2]")
    print(f"  Folha: {folha}")
    print(
        f"  ASTER: {aster_best['id']} ({aster_best['datetime']}), "
        f"cov={aster_best['coverage_fraction']*100.0:.2f}%, "
        f"cloud={aster_best['cloud_cover']:.2f}%"
    )
    print(
        f"  S2   : {s2_best['id']} ({s2_best['datetime']}), "
        f"meta_cloud={s2_best['meta_cloud']:.2f}%, "
        f"scl_cloud_frac={s2_best['scl_cloud_frac']:.4f}"
    )
    print(
        f"  Δt (S2 - ASTER) = "
        f"{abs((item_datetime(s2_item).date() - aster_dt.date()).days)} dias"
    )

    # 4) Inspeção visual dos recortes
    folha_geom = get_folha_geom_geojson(folha)

    inspect_aster_clip(aster_item, folha_geom)
    inspect_s2_clip(s2_item, folha_geom)


# ----------------------------------------------------------------------
# EXECUÇÃO (RODAR DIRETO NO JUPYTER)
# ----------------------------------------------------------------------
if __name__ == "__main__":
    # Em notebook, basta rodar esta célula:
    main()


[PIPELINE] Folha alvo: SB21_ZA_II2_NW
[ASTER] Folha 'SB21_ZA_II2_NW' carregada. Área WGS84 (aprox): 0.015625 (graus²)
[ASTER] Buscando itens em 'aster-l1t'...
[ASTER] Total de itens retornados (antes de filtros locais): 39

[ASTER] Itens candidatos (ordenados):
  id=AST_L1T_00307292004141220_20150505110335, datetime=2004-07-29T14:12:20.641000+00:00, delta_days=1447, coverage=100.00%, cloud=0.00%

[ASTER] Melhor item:
  id=AST_L1T_00307292004141220_20150505110335
  datetime=2004-07-29T14:12:20.641000+00:00
  delta_days=1447
  coverage=100.00%
  cloud=0.00%

[PIPELINE] Data de referência para S2 = data da ASTER selecionada: 2004-07-29T14:12:20.641000+00:00
[S2] Folha 'SB21_ZA_II2_NW' carregada. Área WGS84 (aprox): 0.015625 (graus²)
[S2] Buscando itens em 'sentinel-2-l2a'...
[S2] Total de itens retornados (antes de filtros locais): 5
[S2] Nenhuma cena passou pelos filtros (SCL + meta_cloud).
[PIPELINE] Nenhuma cena S2 adequada encontrada. Encerrando.
